In [1]:
# ============================================================
# CELL 1 — INSPECT THE 1-SECOND SOLETE FILE WITHOUT LOADING IT
# ============================================================

from pathlib import Path

import h5py


# ------------------------------------------------------------
# 1. PATH TO THE 1-SECOND DATASET
# ------------------------------------------------------------

dataset_path = Path(
    "../solete_dataset/SOLETE_Pombo_1sec.h5"
)


# ------------------------------------------------------------
# 2. CHECK THAT THE FILE EXISTS
# ------------------------------------------------------------

print("Dataset exists:")
print(dataset_path.exists())


# ------------------------------------------------------------
# 3. DISPLAY FILE SIZE
# ------------------------------------------------------------

# Convert the file size from bytes to gigabytes.
#
# This helps us understand how large the dataset is
# before attempting to load anything into memory.
file_size_gb = (
    dataset_path.stat().st_size
    / (1024 ** 3)
)

print("\nFile size:")
print(f"{file_size_gb:.2f} GB")


# ------------------------------------------------------------
# 4. INSPECT THE HDF5 STRUCTURE
# ------------------------------------------------------------

# Opening an HDF5 file does NOT automatically load all of
# its data into RAM.
#
# Here we only inspect the structure and metadata.
with h5py.File(dataset_path, "r") as f:

    print("\nTop-level HDF5 groups:")
    print(list(f.keys()))

    # SOLETE measurements are stored inside the DATA group.
    data = f["DATA"]

    print("\nDatasets inside DATA:")
    print(list(data.keys()))

    # Display the shape and datatype of each stored dataset.
    #
    # We are NOT using [:] here because that would read
    # the complete dataset into memory.
    print("\nDataset shapes and data types:")

    for name in data.keys():
        print(
            f"{name}: "
            f"shape={data[name].shape}, "
            f"dtype={data[name].dtype}"
        )

Dataset exists:
True

File size:
3.53 GB

Top-level HDF5 groups:
['DATA']

Datasets inside DATA:
['axis0', 'axis1', 'block0_items', 'block0_values']

Dataset shapes and data types:
axis0: shape=(11,), dtype=|S17
axis1: shape=(39484801,), dtype=int64
block0_items: shape=(11,), dtype=|S17
block0_values: shape=(39484801, 11), dtype=float64


In [2]:
# ============================================================
# CELL 2 — VERIFY 1-SECOND TEMPORAL STRUCTURE SAFELY
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# We do NOT load the complete timestamp array.
#
# Instead, we read:
#   - first 10 timestamps
#   - last 10 timestamps
#
# This is enough to verify the timestamp representation
# and the expected 1-second sampling locally.
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    data = f["DATA"]

    # Read column names from metadata.
    columns = [
        name.decode("utf-8")
        for name in data["axis0"][:]
    ]

    # Read only a tiny timestamp slice from the beginning.
    first_timestamps_raw = data["axis1"][:10]

    # Read only a tiny timestamp slice from the end.
    last_timestamps_raw = data["axis1"][-10:]


# ------------------------------------------------------------
# Convert timestamps from nanoseconds since Unix epoch
# into timezone-aware UTC timestamps.
# ------------------------------------------------------------

first_timestamps = pd.to_datetime(
    first_timestamps_raw,
    unit="ns",
    utc=True
)

last_timestamps = pd.to_datetime(
    last_timestamps_raw,
    unit="ns",
    utc=True
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("Column names:")
for column in columns:
    print("-", column)

print("\nFirst 10 timestamps:")
print(first_timestamps)

print("\nLast 10 timestamps:")
print(last_timestamps)

print("\nFirst timestamp:")
print(first_timestamps[0])

print("\nLast timestamp:")
print(last_timestamps[-1])


# ------------------------------------------------------------
# CHECK THE LOCAL SAMPLING INTERVAL
# ------------------------------------------------------------

print("\nDifferences between first timestamps:")
print(
    pd.Series(first_timestamps)
    .diff()
    .dropna()
    .value_counts()
)

print("\nDifferences between last timestamps:")
print(
    pd.Series(last_timestamps)
    .diff()
    .dropna()
    .value_counts()
)

Column names:
- TEMPERATURE[degC]
- HUMIDITY[%]
- WIND_SPEED[m1s]
- WIND_DIR[deg]
- GHI[kW1m2]
- POA Irr[kW1m2]
- P_Gaia[kW]
- P_Solar[kW]
- Pressure[mbar]
- Azimuth[deg]
- Elevation[deg]

First 10 timestamps:
DatetimeIndex(['2018-11-17 00:00:00+00:00', '2018-11-17 00:00:01+00:00',
               '2018-11-17 00:00:02+00:00', '2018-11-17 00:00:03+00:00',
               '2018-11-17 00:00:04+00:00', '2018-11-17 00:00:05+00:00',
               '2018-11-17 00:00:06+00:00', '2018-11-17 00:00:07+00:00',
               '2018-11-17 00:00:08+00:00', '2018-11-17 00:00:09+00:00'],
              dtype='datetime64[ns, UTC]', freq=None)

Last 10 timestamps:
DatetimeIndex(['2018-09-12 23:59:50+00:00', '2018-09-12 23:59:51+00:00',
               '2018-09-12 23:59:52+00:00', '2018-09-12 23:59:53+00:00',
               '2018-09-12 23:59:54+00:00', '2018-09-12 23:59:55+00:00',
               '2018-09-12 23:59:56+00:00', '2018-09-12 23:59:57+00:00',
               '2018-09-12 23:59:58+00:00', '2018-09-12 2

In [3]:
# ============================================================
# CELL 3 — SCAN COMPLETE 1-SECOND TIMESTAMP STRUCTURE
# ============================================================

import numpy as np
import pandas as pd
import h5py


# ------------------------------------------------------------
# WHY WE ARE DOING THIS
# ------------------------------------------------------------
#
# The first timestamp in the file was:
#
#     2018-11-17
#
# while the last timestamp was:
#
#     2018-09-12
#
# Therefore, the file is clearly NOT stored as one simple
# chronological sequence.
#
# We now scan the timestamp array in chunks.
#
# This allows us to examine all ~39 million timestamps
# without loading the full timestamp array into memory.
#
# We will calculate:
#
#   - true minimum timestamp
#   - true maximum timestamp
#   - number of normal 1-second transitions
#   - number of backward jumps
#   - number of forward gaps
#   - number of consecutive duplicate timestamps
#   - examples of unusual transitions
#
# NOTE:
# This checks CONSECUTIVE timestamp behavior.
# It does not yet test whether the same timestamp appears
# elsewhere in a different block.


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

# Number of timestamps read into memory at one time.
#
# 1,000,000 int64 values require only about 8 MB of
# raw memory, so this is safe for our machine.
chunk_size = 1_000_000

# SOLETE timestamps are stored in nanoseconds.
one_second_ns = 1_000_000_000


# ------------------------------------------------------------
# VARIABLES USED DURING THE SCAN
# ------------------------------------------------------------

global_min = None
global_max = None

normal_one_second = 0
backward_jumps = 0
forward_gaps = 0
consecutive_duplicates = 0

# Store only a few unusual transitions so that
# notebook output remains readable.
unusual_examples = []

previous_last_timestamp = None


# ------------------------------------------------------------
# SCAN THE COMPLETE TIMESTAMP ARRAY
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    timestamps_ds = f["DATA"]["axis1"]

    total_timestamps = len(timestamps_ds)

    for start in range(0, total_timestamps, chunk_size):

        end = min(
            start + chunk_size,
            total_timestamps
        )

        # Read only this timestamp chunk.
        chunk = timestamps_ds[start:end]


        # ----------------------------------------------------
        # GLOBAL MINIMUM / MAXIMUM
        # ----------------------------------------------------

        chunk_min = chunk.min()
        chunk_max = chunk.max()

        if global_min is None or chunk_min < global_min:
            global_min = chunk_min

        if global_max is None or chunk_max > global_max:
            global_max = chunk_max


        # ----------------------------------------------------
        # INCLUDE THE TRANSITION BETWEEN CHUNKS
        # ----------------------------------------------------
        #
        # np.diff() only compares values inside one chunk.
        #
        # Therefore, when this is not the first chunk,
        # prepend the final timestamp from the previous
        # chunk so that no boundary transition is missed.

        if previous_last_timestamp is not None:

            values_for_diff = np.concatenate(
                (
                    np.array([previous_last_timestamp]),
                    chunk
                )
            )

        else:

            values_for_diff = chunk


        # Calculate consecutive timestamp differences.
        differences = np.diff(values_for_diff)


        # ----------------------------------------------------
        # COUNT DIFFERENT TYPES OF TRANSITIONS
        # ----------------------------------------------------

        normal_one_second += np.sum(
            differences == one_second_ns
        )

        backward_jumps += np.sum(
            differences < 0
        )

        forward_gaps += np.sum(
            differences > one_second_ns
        )

        consecutive_duplicates += np.sum(
            differences == 0
        )


        # ----------------------------------------------------
        # SAVE A FEW UNUSUAL TRANSITIONS
        # ----------------------------------------------------

        unusual_positions = np.where(
            differences != one_second_ns
        )[0]

        for position in unusual_positions:

            if len(unusual_examples) >= 20:
                break

            before = values_for_diff[position]
            after = values_for_diff[position + 1]

            unusual_examples.append(
                (
                    before,
                    after,
                    (after - before) / one_second_ns
                )
            )


        # Remember the final timestamp for the next chunk.
        previous_last_timestamp = chunk[-1]


# ------------------------------------------------------------
# CONVERT TRUE DATE RANGE TO READABLE TIMESTAMPS
# ------------------------------------------------------------

true_start = pd.to_datetime(
    global_min,
    unit="ns",
    utc=True
)

true_end = pd.to_datetime(
    global_max,
    unit="ns",
    utc=True
)


# ------------------------------------------------------------
# DISPLAY SUMMARY
# ------------------------------------------------------------

print("Total timestamps:")
print(total_timestamps)

print("\nTrue minimum timestamp:")
print(true_start)

print("\nTrue maximum timestamp:")
print(true_end)

print("\nNormal 1-second transitions:")
print(normal_one_second)

print("\nBackward jumps:")
print(backward_jumps)

print("\nForward gaps greater than 1 second:")
print(forward_gaps)

print("\nConsecutive duplicate timestamps:")
print(consecutive_duplicates)


# ------------------------------------------------------------
# DISPLAY EXAMPLES OF UNUSUAL TRANSITIONS
# ------------------------------------------------------------

print("\nExamples of unusual consecutive transitions:")

for before, after, difference_seconds in unusual_examples:

    before_time = pd.to_datetime(
        before,
        unit="ns",
        utc=True
    )

    after_time = pd.to_datetime(
        after,
        unit="ns",
        utc=True
    )

    print(
        f"{before_time}  ->  {after_time}"
        f"   ({difference_seconds:.0f} seconds)"
    )

Total timestamps:
39484801

True minimum timestamp:
2018-06-01 00:00:00+00:00

True maximum timestamp:
2019-09-01 00:00:00+00:00

Normal 1-second transitions:
39484344

Backward jumps:
239

Forward gaps greater than 1 second:
217

Consecutive duplicate timestamps:
0

Examples of unusual consecutive transitions:
2018-11-18 00:00:00+00:00  ->  2019-01-01 00:00:00+00:00   (3801600 seconds)
2019-01-02 00:00:00+00:00  ->  2018-09-17 00:00:00+00:00   (-9244800 seconds)
2018-09-18 00:00:00+00:00  ->  2018-09-23 00:00:00+00:00   (432000 seconds)
2018-09-24 00:00:00+00:00  ->  2018-08-14 00:00:00+00:00   (-3542400 seconds)
2018-08-15 00:00:00+00:00  ->  2019-06-27 00:00:00+00:00   (27302400 seconds)
2019-06-28 00:00:00+00:00  ->  2019-04-01 00:00:00+00:00   (-7603200 seconds)
2019-04-02 00:00:00+00:00  ->  2018-08-30 00:00:00+00:00   (-18576000 seconds)
2018-08-31 00:00:00+00:00  ->  2019-05-05 00:00:00+00:00   (21340800 seconds)
2019-05-06 00:00:00+00:00  ->  2018-09-10 00:00:00+00:00   (-2056

In [4]:
# ============================================================
# CELL 4 — CHARACTERIZE CONTINUOUS 1-SECOND BLOCKS
# ============================================================

import numpy as np
import pandas as pd
import h5py


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 3 showed:
#
#   - 239 backward jumps
#   - 217 forward gaps
#
# Therefore:
#
#   239 + 217 = 456 boundaries
#
# which implies 457 continuous blocks.
#
# We now determine the SIZE of those blocks.
#
# If most blocks contain approximately one day's worth
# of seconds (86,400 observations), it would strongly
# suggest that the HDF5 file stores daily chunks in a
# shuffled/non-chronological order.
#
# We read timestamps in chunks so that the complete
# 39-million-row timestamp array is never loaded at once.


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

chunk_size = 1_000_000

one_second_ns = 1_000_000_000


# ------------------------------------------------------------
# STORAGE FOR BLOCK INFORMATION
# ------------------------------------------------------------

block_lengths = []

current_block_length = 0

previous_timestamp = None


# ------------------------------------------------------------
# SCAN TIMESTAMPS
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    timestamps_ds = f["DATA"]["axis1"]

    total_timestamps = len(timestamps_ds)

    for start in range(0, total_timestamps, chunk_size):

        end = min(
            start + chunk_size,
            total_timestamps
        )

        chunk = timestamps_ds[start:end]


        # Process each chunk while preserving the transition
        # from the previous chunk.
        if previous_timestamp is not None:

            values = np.concatenate(
                (
                    np.array([previous_timestamp]),
                    chunk
                )
            )

            # The first value belongs to the previous chunk,
            # so we only use it for calculating the boundary
            # difference.
            differences = np.diff(values)

        else:

            differences = np.diff(chunk)


        # ----------------------------------------------------
        # FIND NON-1-SECOND TRANSITIONS
        # ----------------------------------------------------

        break_positions = np.where(
            differences != one_second_ns
        )[0]


        # ----------------------------------------------------
        # CALCULATE BLOCK LENGTHS
        # ----------------------------------------------------
        #
        # Rather than storing the actual observations,
        # we only count how many rows belong to each
        # continuous block.

        segment_start = 0

        for position in break_positions:

            segment_length = (
                position
                - segment_start
                + 1
            )

            current_block_length += segment_length

            block_lengths.append(
                current_block_length
            )

            current_block_length = 0

            segment_start = position + 1


        # Number of observations remaining after the final
        # break in this chunk.
        remaining = (
            len(differences)
            - segment_start
        )

        current_block_length += remaining


        # Remember the last timestamp so that the next
        # chunk boundary can be checked correctly.
        previous_timestamp = chunk[-1]


# ------------------------------------------------------------
# ACCOUNT FOR THE FINAL OBSERVATION / BLOCK
# ------------------------------------------------------------

current_block_length += 1

block_lengths.append(
    current_block_length
)


# ------------------------------------------------------------
# SUMMARIZE BLOCK LENGTHS
# ------------------------------------------------------------

block_lengths = pd.Series(
    block_lengths,
    name="observations_per_block"
)

print("Number of continuous blocks:")
print(len(block_lengths))

print("\nBlock-length summary:")
print(block_lengths.describe())

print("\nMost common block lengths:")
print(block_lengths.value_counts().head(10))

print("\nExpected observations in one 24-hour day:")
print(24 * 60 * 60)

Number of continuous blocks:
457

Block-length summary:
count      457.000000
mean     86400.002188
std          0.812455
min      86399.000000
25%      86399.000000
50%      86400.000000
75%      86401.000000
max      86401.000000
Name: observations_per_block, dtype: float64

Most common block lengths:
observations_per_block
86400    156
86401    151
86399    150
Name: count, dtype: int64

Expected observations in one 24-hour day:
86400


In [5]:
# ============================================================
# CELL 5 — GLOBAL TIMESTAMP UNIQUENESS AND COMPLETENESS
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Earlier checks showed that the 1-second HDF5 file is stored
# as shuffled daily-sized continuous blocks.
#
# We also found no CONSECUTIVE duplicate timestamps.
#
# However, because the blocks are not stored chronologically,
# the same timestamp could theoretically appear in two
# different blocks.
#
# This cell therefore verifies globally that:
#
#   1. every timestamp lies on the expected 1-second grid,
#   2. no timestamp appears more than once,
#   3. no seconds are missing between the true start and end.
#
# We do this without loading all 39 million timestamps into
# a pandas DataFrame.


# ------------------------------------------------------------
# KNOWN TRUE TEMPORAL RANGE
# ------------------------------------------------------------

start_time = pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC"
)

end_time = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC"
)

start_ns = start_time.value
end_ns = end_time.value

one_second_ns = 1_000_000_000


# ------------------------------------------------------------
# EXPECTED NUMBER OF 1-SECOND TIMESTAMPS
# ------------------------------------------------------------
#
# We include both:
#
#   start timestamp
#   end timestamp
#
# therefore +1.

expected_timestamp_count = (
    (end_ns - start_ns) // one_second_ns
    + 1
)

print("Expected number of timestamps:")
print(expected_timestamp_count)


# ------------------------------------------------------------
# CREATE A BOOLEAN OCCUPANCY ARRAY
# ------------------------------------------------------------
#
# Each position represents one expected second.
#
# False -> timestamp has not been seen
# True  -> timestamp has already been seen
#
# ~39.5 million Boolean values require only about 40 MB,
# which is safe compared with loading the full dataset.

timestamp_seen = np.zeros(
    expected_timestamp_count,
    dtype=bool
)


# ------------------------------------------------------------
# GLOBAL COUNTERS
# ------------------------------------------------------------

duplicate_count = 0
off_grid_count = 0
out_of_range_count = 0

chunk_size = 1_000_000


# ------------------------------------------------------------
# SCAN COMPLETE TIMESTAMP DATASET
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    timestamps_ds = f["DATA"]["axis1"]

    total_rows = len(timestamps_ds)

    for start in range(
        0,
        total_rows,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_rows
        )

        chunk = timestamps_ds[start:end]


        # ----------------------------------------------------
        # CHECK WHETHER TIMESTAMPS FALL INSIDE EXPECTED RANGE
        # ----------------------------------------------------

        in_range = (
            (chunk >= start_ns)
            &
            (chunk <= end_ns)
        )

        out_of_range_count += (
            (~in_range).sum()
        )


        # Work only with timestamps inside the valid range.
        valid_chunk = chunk[in_range]


        # ----------------------------------------------------
        # CHECK WHETHER EACH TIMESTAMP LIES EXACTLY ON THE
        # 1-SECOND GRID
        # ----------------------------------------------------

        offsets_ns = (
            valid_chunk
            - start_ns
        )

        on_grid = (
            offsets_ns % one_second_ns
            == 0
        )

        off_grid_count += (
            (~on_grid).sum()
        )


        # Keep only valid 1-second-grid timestamps.
        offsets_seconds = (
            offsets_ns[on_grid]
            // one_second_ns
        ).astype(np.int64)


        # ----------------------------------------------------
        # CHECK FOR TIMESTAMPS SEEN EARLIER
        # ----------------------------------------------------

        duplicate_count += (
            timestamp_seen[
                offsets_seconds
            ].sum()
        )


        # Mark these timestamps as observed.
        timestamp_seen[
            offsets_seconds
        ] = True


# ------------------------------------------------------------
# COUNT MISSING TIMESTAMPS
# ------------------------------------------------------------

missing_timestamp_count = (
    (~timestamp_seen).sum()
)


# ------------------------------------------------------------
# DISPLAY FINAL RESULTS
# ------------------------------------------------------------

print("\nActual rows in HDF5 file:")
print(total_rows)

print("\nDuplicate timestamps across complete file:")
print(duplicate_count)

print("\nMissing expected timestamps:")
print(missing_timestamp_count)

print("\nTimestamps outside expected date range:")
print(out_of_range_count)

print("\nTimestamps not aligned to exact 1-second grid:")
print(off_grid_count)

print("\nComplete one-second coverage?")
print(
    duplicate_count == 0
    and missing_timestamp_count == 0
    and out_of_range_count == 0
    and off_grid_count == 0
)

Expected number of timestamps:
39484801

Actual rows in HDF5 file:
39484801

Duplicate timestamps across complete file:
0

Missing expected timestamps:
0

Timestamps outside expected date range:
0

Timestamps not aligned to exact 1-second grid:
0

Complete one-second coverage?
True


In [6]:
# ============================================================
# CELL 6 — LOAD ONE 1-SECOND CONTINUOUS BLOCK FOR EDA
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The complete 1-second dataset contains almost 39.5 million
# observations, so we do NOT want to load everything into
# pandas just for exploratory analysis.
#
# Cell 4 showed that the file consists of approximately
# one-day continuous blocks.
#
# Here we:
#
#   1. Find the end of the FIRST continuous block
#   2. Load only that block
#   3. Convert it into a pandas DataFrame
#
# This gives us roughly one day's worth of 1-second data
# (~86,400 rows), which is safe to explore in memory.


one_second_ns = 1_000_000_000


# ------------------------------------------------------------
# 1. FIND THE END OF THE FIRST CONTINUOUS BLOCK
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    data = f["DATA"]

    timestamps_ds = data["axis1"]

    # Read only enough timestamps to cover slightly more
    # than one expected day.
    #
    # 100,000 seconds is about 27.8 hours.
    timestamp_probe = timestamps_ds[:100_000]

    # Calculate consecutive timestamp differences.
    differences = np.diff(timestamp_probe)

    # Find the first point where the timestamp difference
    # is NOT exactly one second.
    first_break_positions = np.where(
        differences != one_second_ns
    )[0]

    # The first break marks the end of the first block.
    first_block_end = (
        first_break_positions[0] + 1
    )

    # Read only the first continuous block.
    timestamps_raw = timestamps_ds[
        :first_block_end
    ]

    values = data["block0_values"][
        :first_block_end,
        :
    ]

    # Read column names.
    columns = [
        name.decode("utf-8")
        for name in data["axis0"][:]
    ]


# ------------------------------------------------------------
# 2. CONVERT TIMESTAMPS
# ------------------------------------------------------------

timestamps = pd.to_datetime(
    timestamps_raw,
    unit="ns",
    utc=True
)


# ------------------------------------------------------------
# 3. CREATE SAMPLE DATAFRAME
# ------------------------------------------------------------

df_1sec_sample = pd.DataFrame(
    values,
    columns=columns,
    index=timestamps
)

df_1sec_sample.index.name = "Timestamp"


# ------------------------------------------------------------
# 4. DISPLAY BASIC INFORMATION
# ------------------------------------------------------------

print("Sample block shape:")
print(df_1sec_sample.shape)

print("\nSample block start:")
print(df_1sec_sample.index.min())

print("\nSample block end:")
print(df_1sec_sample.index.max())

print("\nSample duration:")
print(
    df_1sec_sample.index.max()
    - df_1sec_sample.index.min()
)

print("\nFirst observations:")
display(df_1sec_sample.head())

print("\nLast observations:")
display(df_1sec_sample.tail())

Sample block shape:
(86401, 11)

Sample block start:
2018-11-17 00:00:00+00:00

Sample block end:
2018-11-18 00:00:00+00:00

Sample duration:
1 days 00:00:00

First observations:


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-11-17 00:00:00+00:00,14.0,1.6,9.4,376.0,0.0,0.0,0.0,0.0,3000.0,0.0,0.0
2018-11-17 00:00:01+00:00,14.0,1.6,8.8,358.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 00:00:02+00:00,14.0,1.6,6.8,306.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 00:00:03+00:00,14.0,1.6,5.8,402.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 00:00:04+00:00,14.0,1.6,5.8,368.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0



Last observations:


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-11-17 23:59:56+00:00,7.2,1.8,2.2,404.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 23:59:57+00:00,7.4,1.8,2.2,406.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 23:59:58+00:00,7.2,1.8,2.0,406.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-17 23:59:59+00:00,7.2,1.8,2.0,406.0,0.0,0.0,0.0,0.0,2000.0,0.0,0.0
2018-11-18 00:00:00+00:00,7.4,1.8,2.2,406.0,0.0,0.0,0.0,0.0,3000.0,0.0,0.0


In [7]:
# ============================================================
# CELL 7 — SUMMARY OF THE 1-SECOND SAMPLE BLOCK
# ============================================================

# This cell performs a lightweight EDA on the one-day
# 1-second sample that we loaded in Cell 5.
#
# We do NOT assume that this single day represents the
# complete 1-second dataset.
#
# The purpose is only to confirm whether the same types
# of values observed at coarser resolutions are already
# present in the high-resolution data.


# ------------------------------------------------------------
# 1. BASIC STATISTICAL SUMMARY
# ------------------------------------------------------------

print("Statistical summary of the sample block:")

display(
    df_1sec_sample
    .describe()
    .T
)


# ------------------------------------------------------------
# 2. KEY RANGE CHECKS
# ------------------------------------------------------------

sample_checks = {
    "Missing values": (
        df_1sec_sample.isna().sum().sum()
    ),

    "Wind direction > 360°": (
        df_1sec_sample["WIND_DIR[deg]"] > 360
    ).sum(),

    "Pressure > 1100 mbar": (
        df_1sec_sample["Pressure[mbar]"] > 1100
    ).sum(),

    "Gaia power > 12 kW": (
        df_1sec_sample["P_Gaia[kW]"] > 12
    ).sum(),

    "Solar power < 0": (
        df_1sec_sample["P_Solar[kW]"] < 0
    ).sum(),

    "Wind speed < 0": (
        df_1sec_sample["WIND_SPEED[m1s]"] < 0
    ).sum(),
}

print("\nKey range checks for the sample block:")

display(
    pd.Series(sample_checks)
)

Statistical summary of the sample block:


,count,mean,std,min,25%,50%,75%,max
TEMPERATURE[degC],86401.0,9.687393,3.412008,2.2,6.4,11.0,12.400000,14.200000
HUMIDITY[%],86401.0,1.675636,0.120723,1.4,1.6,1.6,1.800000,2.000000
WIND_SPEED[m1s],86401.0,4.280478,2.445337,1.0,2.0,4.0,6.000000,16.600000
WIND_DIR[deg],86401.0,241.518848,80.296456,0.0,172.0,264.0,308.000000,712.000000
GHI[kW1m2],86401.0,0.057552,0.103204,0.0,0.0,0.0,0.092000,0.560000
POA Irr[kW1m2],86401.0,0.094005,0.218450,0.0,0.0,0.0,0.090000,1.468000
P_Gaia[kW],86401.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
P_Solar[kW],86401.0,0.683954,1.527617,0.0,0.0,0.0,0.691734,9.831133
Pressure[mbar],86401.0,2000.023148,4.811197,2000.0,2000.0,2000.0,2000.000000,3000.000000
Azimuth[deg],86401.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000



Key range checks for the sample block:


Missing values               0
Wind direction > 360°     1153
Pressure > 1100 mbar     86401
Gaia power > 12 kW           0
Solar power < 0              0
Wind speed < 0               0
dtype: int64

In [8]:
# ============================================================
# CELL 8 — LOAD A SECOND 1-SECOND DAY FOR VERIFICATION
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The first sampled block (2018-11-17) showed unusual
# behaviour:
#
#   - P_Gaia = 0 for the whole day
#   - Azimuth = 0 for the whole day
#   - Elevation = 0 for the whole day
#   - Pressure ≈ 2000 mbar for the whole day
#
# Because the 1-second file stores daily blocks in a
# shuffled order, we cannot simply use normal row positions
# to select another calendar day.
#
# Here we search the timestamp axis for:
#
#     2018-06-15 00:00:00 UTC
#
# and then load approximately one complete day starting
# from that timestamp.
#
# We still avoid loading the complete 39-million-row
# measurement matrix into memory.


# ------------------------------------------------------------
# 1. TARGET TIMESTAMP
# ------------------------------------------------------------

target_time = pd.Timestamp(
    "2018-06-15 00:00:00",
    tz="UTC"
)

# Convert to the same nanosecond representation used
# inside the HDF5 timestamp array.
target_ns = target_time.value


# ------------------------------------------------------------
# 2. SEARCH FOR THE TARGET TIMESTAMP IN CHUNKS
# ------------------------------------------------------------

chunk_size = 1_000_000

target_index = None

with h5py.File(dataset_path, "r") as f:

    timestamps_ds = f["DATA"]["axis1"]

    total_rows = len(timestamps_ds)

    for start in range(
        0,
        total_rows,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_rows
        )

        timestamp_chunk = timestamps_ds[
            start:end
        ]

        matches = np.where(
            timestamp_chunk == target_ns
        )[0]

        if len(matches) > 0:

            target_index = (
                start + matches[0]
            )

            break


print("Target timestamp:")
print(target_time)

print("\nRow index in shuffled HDF5 file:")
print(target_index)


# ------------------------------------------------------------
# 3. LOAD ONE DAY STARTING FROM THE TARGET
# ------------------------------------------------------------

# One complete day contains 86,400 seconds.
#
# We include the next boundary observation as well,
# matching the style of our first sample.
rows_to_read = 86_401


with h5py.File(dataset_path, "r") as f:

    data = f["DATA"]

    timestamps_raw = data["axis1"][
        target_index:
        target_index + rows_to_read
    ]

    values = data["block0_values"][
        target_index:
        target_index + rows_to_read,
        :
    ]

    columns = [
        name.decode("utf-8")
        for name in data["axis0"][:]
    ]


# ------------------------------------------------------------
# 4. CREATE DATAFRAME
# ------------------------------------------------------------

timestamps = pd.to_datetime(
    timestamps_raw,
    unit="ns",
    utc=True
)

df_1sec_second_sample = pd.DataFrame(
    values,
    columns=columns,
    index=timestamps
)

df_1sec_second_sample.index.name = "Timestamp"


# ------------------------------------------------------------
# 5. DISPLAY BASIC INFORMATION
# ------------------------------------------------------------

print("\nSample shape:")
print(df_1sec_second_sample.shape)

print("\nFirst timestamp:")
print(df_1sec_second_sample.index[0])

print("\nLast timestamp:")
print(df_1sec_second_sample.index[-1])


# ------------------------------------------------------------
# 6. DISPLAY KEY VARIABLE RANGES
# ------------------------------------------------------------

variables_to_compare = [
    "Pressure[mbar]",
    "WIND_DIR[deg]",
    "P_Gaia[kW]",
    "P_Solar[kW]",
    "Azimuth[deg]",
    "Elevation[deg]"
]

print("\nKey-variable summary:")

display(
    df_1sec_second_sample[
        variables_to_compare
    ]
    .describe()
    .T
)

Target timestamp:
2018-06-15 00:00:00+00:00

Row index in shuffled HDF5 file:
12528092

Sample shape:
(86401, 11)

First timestamp:
2018-06-15 00:00:00+00:00

Last timestamp:
2018-06-16 00:00:00+00:00

Key-variable summary:


,count,mean,std,min,25%,50%,75%,max
Pressure[mbar],86401.0,1000.023148,4.811197,1000.0,1000.0,1000.000000,1000.000000,2000.000000
WIND_DIR[deg],86401.0,252.283145,70.573138,0.0,230.0,274.000000,303.000000,356.000000
P_Gaia[kW],86401.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000
P_Solar[kW],86401.0,1.895996,2.250763,0.0,0.0,0.553238,3.951016,6.322918
Azimuth[deg],86401.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000
Elevation[deg],86401.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000


In [9]:
# ============================================================
# CELL 9 — GLOBAL CHECK OF KEY 1-SECOND VARIABLES
# ============================================================

import h5py
import numpy as np


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Two sampled 1-second days showed:
#
#   P_Gaia     = 0 for the complete day
#   Azimuth    = 0 for the complete day
#   Elevation  = 0 for the complete day
#
# Before deciding whether the 1-second dataset is useful
# for our forecasting experiments, we need to determine
# whether these variables ever contain non-zero values
# anywhere in the complete ~39.5-million-row dataset.
#
# We also inspect the global pressure range because the
# sampled days showed 1000 / 2000 / 3000-like behaviour.
#
# We process the HDF5 file in chunks and never create
# a full pandas DataFrame.


# ------------------------------------------------------------
# COLUMN POSITIONS
# ------------------------------------------------------------

# From the SOLETE column order:
#
#  0  TEMPERATURE
#  1  HUMIDITY
#  2  WIND_SPEED
#  3  WIND_DIR
#  4  GHI
#  5  POA Irr
#  6  P_Gaia
#  7  P_Solar
#  8  Pressure
#  9  Azimuth
# 10  Elevation

column_indices = {
    "P_Gaia[kW]": 6,
    "Pressure[mbar]": 8,
    "Azimuth[deg]": 9,
    "Elevation[deg]": 10,
}


# ------------------------------------------------------------
# SCAN SETTINGS
# ------------------------------------------------------------

chunk_size = 1_000_000


# ------------------------------------------------------------
# STORAGE FOR GLOBAL RESULTS
# ------------------------------------------------------------

results = {}

for variable in column_indices:

    results[variable] = {
        "nonzero_count": 0,
        "min": np.inf,
        "max": -np.inf,
    }


# ------------------------------------------------------------
# SCAN COMPLETE FILE
# ------------------------------------------------------------

with h5py.File(dataset_path, "r") as f:

    values_ds = f["DATA"]["block0_values"]

    total_rows = values_ds.shape[0]

    for start in range(
        0,
        total_rows,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_rows
        )

        # Read each selected column only for this chunk.
        for variable, column_index in column_indices.items():

            values = values_ds[
                start:end,
                column_index
            ]

            results[variable]["nonzero_count"] += np.count_nonzero(
                values
            )

            results[variable]["min"] = min(
                results[variable]["min"],
                values.min()
            )

            results[variable]["max"] = max(
                results[variable]["max"],
                values.max()
            )


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("Total rows:")
print(total_rows)

for variable, result in results.items():

    print(f"\n{variable}")

    print("  Non-zero observations:")
    print(" ", result["nonzero_count"])

    print("  Minimum:")
    print(" ", result["min"])

    print("  Maximum:")
    print(" ", result["max"])

Total rows:
39484801

P_Gaia[kW]
  Non-zero observations:
  134890
  Minimum:
  0.0
  Maximum:
  12.312999999999999

Pressure[mbar]
  Non-zero observations:
  39484801
  Minimum:
  991.599975585938
  Maximum:
  3000.0

Azimuth[deg]
  Non-zero observations:
  32600
  Minimum:
  -65.0879974365234
  Maximum:
  55.7190017700195

Elevation[deg]
  Non-zero observations:
  26880
  Minimum:
  0.0
  Maximum:
  12.9870004653931


## 1-Second EDA — Main Findings

The 1-second SOLETE dataset was examined using memory-efficient HDF5
operations rather than loading the complete dataset into pandas.

### Dataset Size

- The HDF5 file is approximately **3.53 GB**.
- It contains **39,484,801 observations** and **11 measured variables**.
- The true temporal coverage is:
  - **2018-06-01 00:00:00 UTC**
  - **2019-09-01 00:00:00 UTC**

### Timestamp Organization

The observations are not stored as one globally chronological sequence.

A complete timestamp scan found:

- **39,484,344 normal one-second transitions**
- **239 backward jumps**
- **217 forward jumps**
- **0 consecutive duplicate timestamps**

The 456 non-one-second boundaries divide the file into **457 continuous
blocks**.

The block lengths are approximately one day:

- Minimum: **86,399 observations**
- Median: **86,400 observations**
- Maximum: **86,401 observations**

This indicates that the 1-second HDF5 file is effectively organized as
daily-sized continuous blocks stored in a non-chronological order.

### Global Timestamp Completeness

A second global validation checked the entire expected one-second time grid.

Results:

- Expected timestamps: **39,484,801**
- Actual HDF5 rows: **39,484,801**
- Duplicate timestamps across the complete file: **0**
- Missing expected timestamps: **0**
- Timestamps outside the expected date range: **0**
- Timestamps not aligned to the exact one-second grid: **0**

Therefore, the timestamp axis is **complete and unique**, despite the
non-chronological block ordering.

This is an important distinction:

> The 1-second data are not temporally incomplete; they are temporally
> complete but stored in shuffled daily-sized blocks.

Forecasting sequences must therefore not be constructed directly from the
stored row order without first restoring chronological ordering.

### Sample-Level Observations

Two individual 1-second days were inspected.

The sampled blocks showed that some variables can remain constant or zero
for an entire day.

For example:

- `P_Gaia[kW]` was zero throughout both inspected days.
- `Azimuth[deg]` and `Elevation[deg]` were also zero throughout both
  inspected days.
- Pressure showed unusual values such as approximately **1000, 2000,
  and 3000 mbar** depending on the sampled block and daily boundary.
- Wind-direction values above 360 degrees were already present in the
  1-second source data.

These findings demonstrate that a single sampled day is not sufficient to
characterize the complete 1-second dataset.

### Global Variable Check

A complete chunked scan of selected variables produced the following
results.

#### `P_Gaia[kW]`

- Non-zero observations: **134,890**
- Non-zero proportion: approximately **0.342%**
- Minimum: **0 kW**
- Maximum: approximately **12.313 kW**

#### `Pressure[mbar]`

- Non-zero observations: **39,484,801**
- Non-zero proportion: **100%**
- Minimum: approximately **991.6 mbar**
- Maximum: **3000 mbar**

#### `Azimuth[deg]`

- Non-zero observations: **32,600**
- Non-zero proportion: approximately **0.083%**
- Minimum: approximately **-65.09 degrees**
- Maximum: approximately **55.72 degrees**

#### `Elevation[deg]`

- Non-zero observations: **26,880**
- Non-zero proportion: approximately **0.068%**
- Minimum: **0 degrees**
- Maximum: approximately **12.99 degrees**

### Important Interpretation

`P_Gaia`, `Azimuth`, and `Elevation` are not globally zero, but their
non-zero observations are extremely sparse relative to the approximately
39.5 million 1-second records.

This differs substantially from their behaviour in the 1-minute and
5-minute datasets.

The pressure and wind-direction irregularities are already present at the
1-second level. Therefore, the corresponding anomalies observed at coarser
temporal resolutions cannot be attributed solely to temporal aggregation.

### Suitability for Forecasting

The 1-second dataset contains valuable high-frequency information, but its:

- very large size,
- shuffled daily-block organization,
- sparse behaviour of several important variables,
- and source-level representation irregularities

make it less convenient for the main multivariable forecasting experiments.

No final temporal resolution is selected in this notebook.

The suitability of 1-second, 1-minute, 5-minute, and 60-minute data will be
evaluated together in the cross-resolution comparison.

### Data-Handling Decision

No source observations were modified.

The original HDF5 file remains unchanged.

All large-scale checks were performed using chunked HDF5 reads to avoid
unnecessary memory usage and to preserve the original source structure.

### Overall 1-Second Assessment

The 1-second SOLETE dataset has a **complete and unique temporal grid**, but
its row ordering and variable behaviour differ substantially from the
cleaner 1-minute and 5-minute processed datasets.

For this reason, it is useful for source-level diagnostics and
cross-resolution analysis, but its suitability for the main forecasting
pipeline should be judged comparatively rather than assumed.

### Next Step

A dedicated cross-resolution comparison will combine the findings from:

- 60-minute SOLETE data
- 5-minute SOLETE data
- 1-minute SOLETE data
- 1-second SOLETE data

The comparison will be used to determine:

- which characteristics are common across resolutions,
- which anomalies are resolution-specific,
- how variable behaviour changes with temporal aggregation,
- and which temporal resolution is most appropriate for the main
  forecasting experiments.